In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import mediapipe as mp
from PIL import Image
from tqdm import tqdm

In [2]:
# ============================================================
# SETUP: MediaPipe and helper functions
# ============================================================

# MediaPipe face mesh module reference
mp_face_mesh = mp.solutions.face_mesh

def ratio_along_axis(point, corner_a, corner_b):
    """
    Ratio of point's position along axis from corner_a to corner_b.
    0.0 = point is at corner_a
    0.5 = point is halfway between
    1.0 = point is at corner_b
    """
    axis = corner_b - corner_a
    axis_len = np.linalg.norm(axis)
    if axis_len < 1e-6:
        return 0.5
    t = np.dot(point - corner_a, axis) / (axis_len ** 2)
    return float(np.clip(t, 0.0, 1.0))

print("Ready")

Ready


In [3]:
# ============================================================
# CEAL FEATURE EXTRACTION
# ============================================================
# Key differences from GazeCapture extraction:
#   - Source: individual full-frame JPGs, not tar files
#   - Labels: parsed from filenames (degrees), not from a CSV
#   - MediaPipe runs on full-frame images (better landmarks)
#   - Output: same 7 features, same parquet format
# ============================================================

CEAL_DATA_ROOT = Path("/Volumes/Crucial X10/210/data/columbia_gaze_dataset")

# ---- Step 1: Parse filenames into metadata ----
# Filename format: {subject}_{distance}m_{headpose}P_{gaze_v}V_{gaze_h}H.jpg

pattern = re.compile(
    r"(?P<subject>\d+)_"
    r"(?P<distance>\d+)m_"
    r"(?P<head_pose>-?\d+)P_"
    r"(?P<gaze_v>-?\d+)V_"
    r"(?P<gaze_h>-?\d+)H\.(jpg|jpeg)$",
    re.IGNORECASE
)

rows = []
for subject_dir in sorted(CEAL_DATA_ROOT.iterdir()):
    if not subject_dir.is_dir():
        continue
    for img_path in subject_dir.glob("*.jpg"):
        match = pattern.match(img_path.name)
        if not match:
            continue
        meta = match.groupdict()
        rows.append({
            "path": str(img_path),
            "filename": img_path.name,
            "subject": meta["subject"],
            "gaze_v": int(meta["gaze_v"]),
            "gaze_h": int(meta["gaze_h"]),
        })

df_ceal = pd.DataFrame(rows)
print(f"CEAL images found: {len(df_ceal)}")
print(f"Subjects: {df_ceal['subject'].nunique()}")
print(f"\nSample row:")
print(df_ceal.iloc[0])

CEAL images found: 5880
Subjects: 56

Sample row:
path        /Volumes/Crucial X10/210/data/columbia_gaze_da...
filename                             0001_2m_-30P_0V_-10H.jpg
subject                                                  0001
gaze_v                                                      0
gaze_h                                                    -10
Name: 0, dtype: object


In [4]:
# ============================================================
# CEAL FEATURE EXTRACTION
#
# Same 7 features as GazeCapture, but computed on full-frame
# CEAL images instead of 112x112 face crops.
#
# Full-frame images → better landmarks → more accurate features.
#
# Output: parquet file keyed by CEAL filename (not tar key,
# since CEAL doesn't use tars).
#
# Features:
#   1. left_iris_h       — iris horizontal ratio, subject's left eye
#   2. right_iris_h      — iris horizontal ratio, subject's right eye
#   3. iris_h_agreement  — left minus right (binocular consistency)
#   4. head_yaw          — nose horizontal offset from eye center
#   5. head_pitch        — nose vertical offset below eye line
#   6. z_tilt            — z(chin) minus z(forehead)
#   7. z_nose_rel        — z(nose) minus z(forehead)
# ============================================================

CEAL_GEO_OUTPUT = "/Volumes/Crucial X10/210/ceal_geo_features_v1.parquet"

records = []
fail_count = 0

with mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
) as face_mesh:

    for _, row in tqdm(df_ceal.iterrows(), total=len(df_ceal), desc="CEAL features"):
        img = Image.open(row['path']).convert("RGB")
        img_np = np.array(img)
        h, w = img_np.shape[:2]

        results = face_mesh.process(img_np)

        if not results.multi_face_landmarks or len(results.multi_face_landmarks[0].landmark) < 478:
            fail_count += 1
            continue

        lms = results.multi_face_landmarks[0].landmark

        # ---- Helper: landmark index → pixel coords ----
        def lm_px(idx):
            return np.array([lms[idx].x * w, lms[idx].y * h])

        # ---- Features 1 & 2: iris horizontal ratios ----
        left_iris_h  = ratio_along_axis(lm_px(473), lm_px(263), lm_px(362))
        right_iris_h = ratio_along_axis(lm_px(468), lm_px(33),  lm_px(133))

        # ---- Feature 3: binocular agreement ----
        iris_h_agreement = left_iris_h - right_iris_h

        # ---- Features 4 & 5: head pose from 2D landmarks ----
        nose        = lm_px(1)
        left_outer  = lm_px(263)
        right_outer = lm_px(33)
        eye_cx   = (left_outer[0] + right_outer[0]) / 2.0
        eye_cy   = (left_outer[1] + right_outer[1]) / 2.0
        eye_span = abs(left_outer[0] - right_outer[0])

        head_yaw   = (nose[0] - eye_cx) / max(eye_span, 1e-6)
        head_pitch = (nose[1] - eye_cy) / max(eye_span, 1e-6)

        # ---- Features 6 & 7: z-depth signals ----
        z_tilt     = lms[152].z - lms[10].z
        z_nose_rel = lms[1].z   - lms[10].z

        # ---- Key: use filename as identifier ----
        records.append({
            'filename': row['filename'],
            'subject': row['subject'],
            'left_iris_h': left_iris_h,
            'right_iris_h': right_iris_h,
            'iris_h_agreement': iris_h_agreement,
            'head_yaw': head_yaw,
            'head_pitch': head_pitch,
            'z_tilt': z_tilt,
            'z_nose_rel': z_nose_rel,
        })

df_ceal_geo = pd.DataFrame(records)
df_ceal_geo.to_parquet(CEAL_GEO_OUTPUT, index=False)

print(f"\nDone!")
print(f"  Samples with features: {len(df_ceal_geo)}")
print(f"  Failed detections:     {fail_count}")
print(f"  Saved to: {CEAL_GEO_OUTPUT}")

I0000 00:00:1772725800.449000       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
CEAL features: 100%|██████████| 5880/5880 [12:58<00:00,  7.55it/s]



Done!
  Samples with features: 5880
  Failed detections:     0
  Saved to: /Volumes/Crucial X10/210/ceal_geo_features_v1.parquet
